#3 - A partir das extremidades das águas correntes e das águas correntes estimadas, traçar um círculo de 50m;

Desculpa a minha demora aqui... eu tive um problema com meu convênio de saúde e fiquei birutinha. OK, vamos tentar voltar pro corpo, né?  
  
Acho que teríamos que fazer:
* Comparar os sem identificação (SD) entre o drenagem e o correntes estimadas
* Aquele processo do Henrique tanto no drenagem (que ele já fez, mas quando for pro outro github eu vou dar uma mudada)... (? quanto no de correntes estimadas do Elias)
* Pegar os pontinhos no drenaghenrique e fazer um simmetrical difference com os pontos do correntes_estimadas...
* ... aí eu preciso conferir, né (ainda n sei como)

In [1]:
import geopandas as gpd
from os.path import join
from tqdm import tqdm

In [2]:
def testar_gdf(gdf):
    print(
    f'Shape: {gdf.shape};\n'+
    f'\nSample:{gdf.sample()}'
)

Ok, vamos começar do começo!
# A. Comparar drenagem ([GeoSampa](http://wfs.geosampa.prefeitura.sp.gov.br/geoserver/geoportal/wfs?version=1.0.0&request=GetFeature&outputFormat=SHAPE-ZIP&typeName=geoportal:drenagem)) e correntes estimadas ([Elias](https://github.com/sepep-pmsp/siiau/blob/master/projects/urbis/assets/silver/parquet_aguas_correntes_estimadas.py))

In [3]:
drenageo = gpd.read_file(
    join(
        'data',
        'drenagem.zip'
    )
)
print(
    f'Shape: {drenageo.shape};\n'+
    f'\nSample:{drenageo.sample()}'
)

## Como as correntes estimadas tem a id como int, vou transformar a id do drenageo em int tbm
drenageo['cd_identif'].astype('int', copy=False)

Shape: (27611, 15);

Sample:       cd_identif cd_tipo_ac tx_tipo_ac cd_numero_ nm_bairro nm_acident  \
27240     26029.0         ND       None          2       S/B         SD   

       qt_comprim  cd_tipo_cu                nm_tipo_cu nm_via_pro nm_descrit  \
27240   72.356688        11.0  Trecho em estado natural        S/N       None   

                nm_tipo_tr dt_atualiz cd_usuario  \
27240  Trecho a céu aberto 2025-01-03       None   

                                                geometry  
27240  LINESTRING (323510.425 7361206.034, 323521.815...  


0            1
1        26125
2            2
3        26126
4        26127
         ...  
27606    27596
27607    27597
27608    27598
27609    27599
27610    27600
Name: cd_identif, Length: 27611, dtype: int64

Provavelmente eles são iguais mesmo, só que as correntes estimadas são polígonos ao invés de linhas  
# B. Conferir SDs
Vamos conferir quais os 'nm_acidentes' que aparecem e se tem SD ou outra forma de dados vazios nesse recorte

Pelo que eu vi e conversei com o Mauryas: ```Considere os tipos para efeito de continuidade. Só não gere a nascente se esses tipos desconhecidos forem os finais de um curso.```  
Sabe o que isso quer dizer também? Que não dá pra eu separar por nomes igual o Henrique fez...

In [4]:
drenageo.sample(3) # o melhor era aqui ser um drenageo_in_estim, né, mas ok, vamos trabalhar só com o drenageo por enquanto

,cd_identif,cd_tipo_ac,tx_tipo_ac,cd_numero_,nm_bairro,nm_acident,qt_comprim,cd_tipo_cu,nm_tipo_cu,nm_via_pro,nm_descrit,nm_tipo_tr,dt_atualiz,cd_usuario,geometry
3166,2997.0,COR,CORREGO,4,SANTO AMARO,CORREGO MORRO DO S,877.443641,10.0,Trecho canalizado subterrâneo,JOAO DIAS,Córrego Morro do S,Trecho fechado,2025-01-03,None,"LINESTRING (322719.045 7384725.227, 322794.35 ..."
21172,20179.0,ND,None,1,S/B,SD,197.274490,11.0,Trecho em estado natural,S/N,None,Trecho a céu aberto,2025-01-03,None,"LINESTRING (338569.62 7410449.82, 338572.421 7..."
13545,26674.0,ND,None,2,EMBURA,SD,87.865008,11.0,Trecho em estado natural,ENG MARCILAC A EVANG. DE SOUZA,None,Trecho a céu aberto,2025-01-03,None,"LINESTRING (329917.124 7355090.628, 329955.694..."


O que eu acho que dá pra fazer:  
* Dar um buffer, ou nos pontos, ou na linha
* Considerar, da lista com mesmo nome, qual é o que tem dois pontos touching um outro próximo com mesmo nome
* Deverão haver 2 sem toucing, o mais alto no index é considerado o 1
* Faz aquele negócio que o Henrique ensinou, mas precisamos ter certeza de que vai terminar ou em A ou em B
* __A)__ total de linhas com nome -1
* __B)__ total de linhas com nome, mas garantir que nenhum ponto do final encoste em nenhum ponto do inicial (caso numer_de_segmentos>2)

Agora, sobre os SDs, como eu posso resolver?  
Talvez seja uma boa usar shortest line com este, mas não com os pontos...  
ANTES DE TUDO eu vou dar um explore com cores diferentes, já tendo a linha unificada dos nomes traçada. Aí eu consigo ter um panorama